**Study of the source: ODEPA wholesale prices, 2026**

Before publishing a single price, it is worth knowing whether the file ODEPA delivers comes clean. This notebook goes through the 2026 wholesale file with an orderly procedure, step by step, and shows what we found along the way. In each step we say what we are going to check and why, leave the code in sight and explain what came up. When something does not add up, we record it as an anomaly, together with the decision we took and the way the project handles it. The figures in the text refer to the file published up to 30 October 2026. Product names, units and column names stay as ODEPA writes them, in Spanish.

**Loading the file**

We need four libraries: `requests` to download the file, `pandas` to handle it as a table, and `io` and `re` to read it in memory and search for patterns in the text.

In [1]:
# io: read what was downloaded in memory; re: regular expressions; pandas: tables; requests: download.
import io
import re

import pandas as pd
import requests

We download the file straight from ODEPA's data portal, without saving it to disk, and keep it in a table called `raw`. We read every value as text, exactly as it comes, so that nothing gets corrected behind our back.

In [2]:
URL = "https://datos.odepa.gob.cl/dataset/33f10516-acbe-4446-b633-68244b9b6b26/resource/580beca0-e87e-4dd4-9e8a-0bd92773f4a6/download/precio_mayorista_fruta-hortaliza_2026.csv"

# ODEPA's certificate authority (emSign) left certifi's list, so it is trusted explicitly.
response = requests.get(URL, timeout=300, verify="../agent/certs/emsign_root_ca_g1.pem")
response.raise_for_status()
# dtype=str keeps every value exactly as the source wrote it.
raw = pd.read_csv(io.BytesIO(response.content), dtype=str, encoding="utf-8-sig")

print(f"{len(raw):,} rows and {raw.shape[1]} columns were loaded from the 2026 wholesale file.")

155,857 rows and 14 columns were loaded from the 2026 wholesale file.


**How we are going to check it**

With 155,857 rows there is no point in reading the file from top to bottom. What we do is ask it questions, from the most general to the most particular, and let the code count how many rows answer one way or another. There are eight steps:

| Step | Question | What is looked for |
|---|---|---|
| 1. Structure | What does the file hold and in what form? | columns, types and a sample of rows |
| 2. Completeness | Is any data missing? | empty values, days and markets without rows |
| 3. Validity | Do the values make sense? | ranges and consistency between minimum, average and maximum |
| 4. Uniqueness | Are there repeated rows? | rows that share their key |
| 5. Categories | How many ways are there of writing each thing? | units, qualities, varieties and origins |
| 6. Scale | Does one product have prices of very different scales? | price by product and unit |
| 7. Outliers | Are there prices out of the usual? | each price against the median of its group |
| 8. Comparability | Do packages change from one week to the next? | packages of each product by week |

Two small functions help us read the results. `number` writes figures with a comma for thousands, and `show` presents tables without row numbers and with the important column in yellow. They change no data, only how it looks.

In [3]:
# The column with the unit of sale.
UNIT = "Unidad de comercializacion"


# Writes a number to read: thousands with a comma, and two decimals only below 100.
def number(value):
    if not float(value).is_integer() and abs(value) < 100:
        return f"{value:.2f}".rstrip("0").rstrip(".")
    return f"{value:,.0f}"


# A table to read: no row numbers and the given columns highlighted.
def show(table, mark=None):
    if isinstance(table, pd.Series):
        table = table.to_frame()
    view = table.style.format(number, subset=table.select_dtypes("number").columns)
    if table.index.name is None:
        view = view.hide(axis="index")
    if mark:
        view = view.set_properties(
            subset=mark, **{"background-color": "rgba(245, 197, 66, 0.3)", "font-weight": "bold"}
        )
    return view

**Step 1 · What does the file hold?**

Before looking for errors we need to know what to expect from each column, and the simplest way is to look. We take 100 rows at random, always the same ones, so that anyone can repeat the exercise.

In [4]:
# Random sample with a fixed seed, so it can be repeated.
print(f"Random sample of 100 rows, out of a total of {number(len(raw))}.")
raw.sample(100, random_state=7).sort_index()

Random sample of 100 rows, out of a total of 155,857.


,Fecha,ID region,Region,Mercado,Subsector,Producto,Variedad / Tipo,Calidad,Unidad de comercializacion,Origen,Volumen,Precio minimo,Precio maximo,Precio promedio
1717,2026-01-06,8,Región del Biobío,Vega Monumental Concepción,Frutas,Nectarín,Pulpa amarilla,Segunda,$/bandeja 18 kilos granel,Región de O'Higgins,200,"10000,0000","10000,0000","10000,0000"
1757,2026-01-06,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Frutas,Palta,Edranol,Segunda,$/kilo (en caja de 17 kilos),Provincia de Quillota,55,"1600,0000","1600,0000","1600,0000"
4415,2026-01-09,13,Región Metropolitana de Santiago,Mercado Mayorista Lo Valledor de Santiago,Hortalizas y tubérculos,Tomate,Semiduro,Extra,$/bandeja 18 kilos,Región de O'Higgins,500,"12000,0000","12000,0000","12000,0000"
5255,2026-01-12,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Hortalizas y tubérculos,Cebollín,Sin especificar,Segunda,$/paquete 36 unidades,Región Metropolitana,250,"2800,0000","3000,0000","2900,0000"
6853,2026-01-14,13,Región Metropolitana de Santiago,Vega Central Mapocho de Santiago,Hortalizas y tubérculos,Zapallo,Camote,1a nueva(o),$/kilo (volumen en unidades),Región Metropolitana,1240,"400,0000","600,0000","500,0000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144978,2026-10-09,4,Región de Coquimbo,Terminal La Palmera de La Serena,Frutas,Palta,Edranol,Primera,$/kilo (en caja de 17 kilos),Provincia de Limarí,160,"1600,0000","1700,0000","1650,0000"
145474,2026-10-13,5,Región de Valparaíso,Femacal de La Calera,Frutas,Frutilla,Sin especificar,Primera,$/bandeja 7 kilos,Provincia de Melipilla,68,"8000,0000","8000,0000","8000,0000"
150548,2026-10-22,9,Región de La Araucanía,Vega Modelo de Temuco,Hortalizas y tubérculos,Betarraga,Sin especificar,Primera,$/docena de paquetes,Provincia de Cautín,80,"12000,0000","13000,0000","12500,0000"
155083,2026-10-29,4,Región de Coquimbo,Terminal La Palmera de La Serena,Hortalizas y tubérculos,Zapallo,Camote,1a nueva(o),$/kilo (volumen en unidades),Perú,1300,"2300,0000","2400,0000","2350,0000"


Looking at rows helps, but it is not enough for 155,857 of them. So we ask the code for a summary of each column: how many distinct values it has, what its most common value looks like (turning each digit into a 9 and each word into an «a») and what share of its values looks like a number.

In [5]:
# The form of a value: every digit becomes 9 and every run of letters becomes "a".
def form(column):
    forms = raw[column].str.replace(r"\d", "9", regex=True)
    forms = forms.str.replace(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", "a", regex=True)
    return forms.value_counts().index[0]


print(f"Summary of the {raw.shape[1]} columns: distinct values, most common form and share with the form of a number.")
structure = pd.DataFrame(
    {
        "distinct values": raw.nunique(),
        "most common form": [form(column) for column in raw.columns],
        "look like a number (%)": [raw[c].str.fullmatch(r"\d+(,\d+)?").mean() * 100 for c in raw.columns],
    }
).rename_axis("column")
show(structure, mark=["look like a number (%)"])

Summary of the 14 columns: distinct values, most common form and share with the form of a number.


,distinct values,most common form,look like a number (%)
column,,,
Fecha,209,9999-99-99,0
ID region,9,99,100
Region,9,a a a a,0
Mercado,12,a a a a,0
Subsector,2,a a a,0
Producto,81,a,0
Variedad / Tipo,208,a a,0
Calidad,55,a,0
Unidad de comercializacion,148,$/a 99 a,0


The summary makes it clear that everything arrives as text. Five columns have the form of a number: the region identifier, the volume and the three prices. The prices, on top of that, are written with a decimal comma and four decimals, like `4000,0000`.

**Anomaly 1 · Numbers arrive as text.** Prices and volumes come as text, so they cannot be added or compared as they are. At first sight it is not clear either whether the comma separates decimals or thousands. Since every price has exactly four digits after the comma, we read it as a decimal. We convert the values to numbers and keep the original text alongside, in case the conversion ever needs checking.

**Step 2 · Is any data missing?**

Missing data is not always visible at a glance. We check two things: whether there are empty cells and whether whole days are missing. We start with the cells.

In [6]:
# Empty values and edge spaces, by column.
stripped = raw.apply(lambda column: column.str.strip())
print(
    f"{number(int(raw.isna().sum().sum() + (stripped == '').sum().sum()))} empty values and "
    f"{number(int((raw != stripped).sum().sum()))} with edge spaces were found."
)
show(
    pd.DataFrame(
        {
            "empty": raw.isna().sum() + (stripped == "").sum(),
            "edge spaces": (raw != stripped).sum(),
        }
    ).rename_axis("column"),
    mark=["empty"],
)

0 empty values and 0 with edge spaces were found.


,empty,edge spaces
column,,
Fecha,0,0
ID region,0,0
Region,0,0
Mercado,0,0
Subsector,0,0
Producto,0,0
Variedad / Tipo,0,0
Calidad,0,0
Unidad de comercializacion,0,0


A daily file should not skip working days. We look for the days from Monday to Friday that have no rows at all and cross them with Chile's official calendar of public holidays, because a day without data may be a holiday or a publication failure.

In [7]:
# Chilean public holidays on working days, between the first and the last date of the file (official calendar).
HOLIDAYS = {
    "2026-04-03": "Good Friday",
    "2026-05-01": "Labour Day",
    "2026-05-21": "Navy Day",
    "2026-06-29": "Saints Peter and Paul",
    "2026-07-16": "Our Lady of Mount Carmel",
    "2026-09-18": "Independence Day",
    "2026-10-12": "Meeting of Two Worlds",
}
weekdays = pd.bdate_range(raw["Fecha"].min(), raw["Fecha"].max()).strftime("%Y-%m-%d")
missing = sorted(set(weekdays) - set(raw["Fecha"]))
print(
    f"{len(missing)} working days without rows were found; {len(set(missing) & set(HOLIDAYS))} of them are "
    "official public holidays."
)
show(pd.DataFrame({"date": missing, "holiday": [HOLIDAYS.get(day, "—") for day in missing]}))

7 working days without rows were found; 7 of them are official public holidays.


date,holiday
2026-04-03,Good Friday
2026-05-01,Labour Day
2026-05-21,Navy Day
2026-06-29,Saints Peter and Paul
2026-07-16,Our Lady of Mount Carmel
2026-09-18,Independence Day
2026-10-12,Meeting of Two Worlds


Having data on a given day does not mean every market reported that day. We count how many days each market appears.

In [8]:
# Days with rows of each market, against the days of the file.
days = raw.groupby("Mercado")["Fecha"].nunique()
print("Days with data for each market, from fewest to most.")
show(
    pd.DataFrame({"days with rows": days, "% of the days of the file": days / raw["Fecha"].nunique() * 100})
    .rename_axis("market")
    .sort_values("days with rows"),
    mark=["days with rows"],
)

Days with data for each market, from fewest to most.


,days with rows,% of the days of the file
market,,
Comercializadora del Agro de Limarí,84,40.19
Vega Monumental Concepción,168,80.38
Feria Lagunitas de Puerto Montt,194,92.82
Terminal La Palmera de La Serena,206,98.56
Mapocho venta directa de Santiago,207,99.04
Terminal Hortofrutícola Agro Chillán,208,99.52
Agrícola del Norte S.A. de Arica,209,100
Femacal de La Calera,209,100
Macroferia Regional de Talca,209,100


There are no empty cells or stray spaces, and the seven missing working days are all public holidays. What does show up is that some markets report quite a few days fewer than the rest.

**Anomaly 2 · Days and markets without data.** The seven working days without rows are public holidays, such as 1 May and 18 September, so there is nothing to correct there. Markets are a different matter: Comercializadora del Agro de Limarí appears on 84 of the 209 days, and Vega Monumental Concepción on 168. The file does not say whether they were closed or simply did not report. We decided not to fill anything in. The price of each week is computed with the days that exist, and if a whole week goes missing between weeks with data, the load stops.

**Step 3 · Do the values make sense?**

A price at zero, or a minimum above the maximum, would spoil any later calculation. To find them we first turn prices and volume into numbers, always next to the original text, and then count the rows that fail each check.

In [9]:
# Working columns beside the original text; whatever is not a number stays empty and is counted.
for name, column in {
    "low": "Precio minimo",
    "high": "Precio maximo",
    "avg": "Precio promedio",
    "volume": "Volumen",
}.items():
    raw[name] = pd.to_numeric(raw[column].str.replace(",", ".", regex=False), errors="coerce")

numbers = raw[["low", "high", "avg", "volume"]]
checks = {
    "price or volume that is not a number": numbers.isna().any(axis=1).sum(),
    "price or volume at zero or below": (numbers <= 0).any(axis=1).sum(),
    "minimum above maximum": (raw["low"] > raw["high"]).sum(),
    "average outside minimum and maximum": ((raw["avg"] < raw["low"]) | (raw["avg"] > raw["high"])).sum(),
    "date outside 2026": (~raw["Fecha"].str.startswith("2026")).sum(),
}
print(
    f"{len(checks)} checks were applied to the {number(len(raw))} rows; "
    f"{sum(value > 0 for value in checks.values())} of them found rows with problems."
)
show(pd.Series(checks, name="failing rows").rename_axis("check"), mark=["failing rows"])

5 checks were applied to the 155,857 rows; 0 of them found rows with problems.


,failing rows
check,
price or volume that is not a number,0
price or volume at zero or below,0
minimum above maximum,0
average outside minimum and maximum,0
date outside 2026,0


No check fails. There are no values that are not numbers, no prices at zero or below, and no minimum above the maximum. A finer question remains: whether the published average is simply the midpoint between the minimum and the maximum.

In [10]:
# The midpoint between the minimum and the maximum of each row, against its published average.
midpoint = (raw["low"] + raw["high"]) / 2
different = raw[(raw["avg"] - midpoint).abs() >= 0.001]
print(
    f"{number(len(different))} rows were found whose average does not match the midpoint between minimum and maximum."
)
show(different.head(5)[["Producto", "Precio minimo", "Precio maximo", "Precio promedio"]])

25,600 rows were found whose average does not match the midpoint between minimum and maximum.


Producto,Precio minimo,Precio maximo,Precio promedio
Acelga,"17000,0000","18000,0000","17666,6666"
Acelga,"550,0000","600,0000","571,4285"
Ají,"30000,0000","31000,0000","30222,2222"
Ají,"18000,0000","20000,0000","19379,3103"
Ajo,"17000,0000","18000,0000","17428,5714"


**Anomaly 3 · The average is not the midpoint.** In 25,600 rows it is not. Between $17,000 and $18,000, for instance, the published average is $17,666 rather than $17,500. No row falls outside its range, so it does not look like an error; ODEPA may weight what each trader sold, although it does not document it. We use the average as it comes. ODEPA saw the sales of that day and we only have the minimum and the maximum, so recomputing it would mean losing information.

In [11]:
# How many times the maximum is the minimum, row by row.
raw["max_over_min"] = raw["high"] / raw["low"]
wide = raw[raw["max_over_min"] > 2]
print(f"{len(wide)} rows were found whose maximum price is more than twice the minimum.")
show(
    wide.sort_values("max_over_min", ascending=False)[
        ["Fecha", "Producto", "Unidad de comercializacion", "Precio minimo", "Precio maximo", "max_over_min"]
    ],
    mark=["max_over_min"],
)

6 rows were found whose maximum price is more than twice the minimum.


Fecha,Producto,Unidad de comercializacion,Precio minimo,Precio maximo,max_over_min
2026-03-11,Zapallo,$/kilo (volumen en unidades),"300,0000","4000,0000",13.33
2026-07-10,Palta,$/bandeja 10 kilos,"2000,0000","22000,0000",11
2026-09-21,Brócoli,$/unidad,"300,0000","800,0000",2.67
2026-01-02,Choclo,$/unidad,"100,0000","250,0000",2.5
2026-04-15,Brócoli,$/unidad,"200,0000","500,0000",2.5
2026-01-16,Limón,$/bandeja 18 kilos,"7000,0000","15000,0000",2.14


**Anomaly 4 · Minimums and maximums far apart.** Only six rows have a maximum more than twice the minimum, but some stand out: a pumpkin between $300 and $4,000, or a 10-kilo tray of avocado between $2,000 and $22,000, on the same day and in the same market. For the calculation we only use the average price. The minimum and the maximum help us rule out the impossible, and if the average strays too far, the outlier check of step 7 catches it (anomaly 16).

**Step 4 · Are there repeated rows?**

A duplicated row would count the same price twice, but one that only looks duplicated must not be deleted. We look for both: identical rows, and rows that share day, market, product, variety, quality and unit.

In [12]:
# Rows that share their key: day, market, product, variety, quality and unit.
key = ["Fecha", "Mercado", "Producto", "Variedad / Tipo", "Calidad", UNIT]
repeated = raw[raw.duplicated(key, keep=False)]
different_price = (repeated.groupby(key)["avg"].nunique() > 1).sum()
print(
    f"{number(raw.duplicated().sum())} identical rows and {number(len(repeated))} rows sharing their key were found, "
    f"in {number(repeated.groupby(key).ngroups)} groups; in {number(different_price)} of them the prices differ."
)
show(
    repeated.sort_values(key).head(6)[["Fecha", "Mercado", "Producto", UNIT, "Origen", "Volumen", "Precio promedio"]],
    mark=["Origen"],
)

0 identical rows and 18,742 rows sharing their key were found, in 8,948 groups; in 6,336 of them the prices differ.


Fecha,Mercado,Producto,Unidad de comercializacion,Origen,Volumen,Precio promedio
2026-01-02,Femacal de La Calera,Cebolla,$/malla 18 kilos,Provincia de Santiago,90,"11722,2222"
2026-01-02,Femacal de La Calera,Cebolla,$/malla 18 kilos,Provincia de Melipilla,45,"12000,0000"
2026-01-02,Femacal de La Calera,Choclo,$/unidad,Provincia de Quillota,13000,"200,0000"
2026-01-02,Femacal de La Calera,Choclo,$/unidad,Región Metropolitana,17500,"214,8571"
2026-01-02,Femacal de La Calera,Papa,$/saco 25 kilos,Provincia de Quillota,160,"7000,0000"
2026-01-02,Femacal de La Calera,Papa,$/saco 25 kilos,Región Metropolitana,190,"7000,0000"


There is not a single identical row. There are many, though, that share all of the above and differ in just one column, the origin.

**Anomaly 5 · Rows that look repeated.** There are 18,742 of them, and in 6,336 of their groups the prices differ. It is the same product, on the same day and in the same market, but brought from different places. They are not duplicates; they are different sales. We keep them all, and the price of the week weights them by what each one holds.

**Step 5 · How many ways is each thing written?**

Free-text columns are the trickiest. If one thing is written in several ways, or one label lumps different things together, prices stop being comparable. We start by counting how many distinct values each one has.

In [13]:
# Distinct values and most frequent value of the free-text columns.
columns = ["Producto", "Variedad / Tipo", "Calidad", UNIT, "Origen"]
print("Distinct values and most frequent value of the free-text columns.")
categories = pd.DataFrame(
    {
        "distinct values": raw[columns].nunique(),
        "most frequent value": [raw[c].value_counts().index[0] for c in columns],
        "% of the rows": [round(raw[c].value_counts(normalize=True).iloc[0] * 100, 1) for c in columns],
    }
).rename_axis("column")
show(categories, mark=["distinct values"])

Distinct values and most frequent value of the free-text columns.


,distinct values,most frequent value,% of the rows
column,,,
Producto,81,Palta,7.3
Variedad / Tipo,208,Sin especificar,36.3
Calidad,55,Primera,48.9
Unidad de comercializacion,148,$/unidad,8.8
Origen,59,Región de O'Higgins,18


Of the five, the one that matters most for prices is the unit of sale, with 148 forms. To see what they are made of, we separate the type of package, which is the word that follows «$/».

In [14]:
# The type of package is the word that follows «$/».
packaging = raw[UNIT].str.extract(r"^\$/(\w+)")[0]

types = raw.assign(package=packaging).groupby("package").agg(forms=(UNIT, "nunique"), rows=(UNIT, "size"))
print(f"{types.forms.sum()} ways of writing the unit of sale were found, grouped in {len(types)} types of package.")
show(types.sort_values("rows", ascending=False).rename_axis("type of package"), mark=["forms"])

148 ways of writing the unit of sale were found, grouped in 15 types of package.


,forms,rows
type of package,,
caja,56,"43,522"
bandeja,22,"39,357"
malla,14,"16,143"
unidad,1,"13,733"
kilo,10,"11,172"
saco,7,"9,192"
bins,4,"8,891"
docena,10,"6,816"
paquete,11,"4,634"


**Anomaly 6 · The unit of sale is written in 148 forms.** Each form combines a package (tray, box, mesh bag, sack, bin, bunch, dozen…) with a weight or a quantity, and sometimes one more word, like `granel` or `empedrada`. As long as we do not know what each package holds, the prices of one product cannot be compared. That is why every form was reviewed by hand and written down in a list, `etl/reviewed_units.csv`. If a new form appears, the load stops until someone reviews it. Today every form of 2026 is on the list.

With the reviewed list we can add to each row how much its package holds, in kilos or in units. The forms the list leaves without a reading are the ones that do not allow a price per kilo.

In [15]:
# The list reviewed by hand says how much each form holds; those without a safe reading stay empty.
reviewed = pd.read_csv("../etl/reviewed_units.csv").rename(columns={"unit": UNIT})
raw = raw.merge(reviewed, on=UNIT, how="left")
no_reading = raw[raw["expected_unit"].isna()]
print(
    f"The {number(raw[UNIT].nunique())} forms were matched against the reviewed list: "
    f"{number((~raw[UNIT].isin(reviewed[UNIT])).sum())} are missing from it."
)
print(
    f"{number(len(no_reading))} rows, in {no_reading[UNIT].nunique()} forms, were found without a safe reading of their content."
)
show(no_reading[UNIT].value_counts().head(8).rename_axis("unit").to_frame("rows"))

The 148 forms were matched against the reviewed list: 0 are missing from it.
8,276 rows, in 15 forms, were found without a safe reading of their content.


,rows
unit,
$/docena de atados,"2,465"
$/docena de matas,"2,120"
$/caja 36 atados,"1,137"
"$/atado 0,5 a 1 kilo",693
$/docena de paquetes,664
$/paquete,324
"$/atado 1,5 a 2 kilos",209
$/atado 300 a 500 gramos,189


**Anomaly 7 · Units without a defined weight.** There are 8,276 rows in that situation. Some units do not say how much they weigh, like `docena de atados`, and others only give a range, like `atado 0,5 a 1 kilo`; a bunch may weigh half a kilo or a whole one. We do not make up a weight. These rows are published without a price per kilo and stay flagged, without being deleted, because a missing price is better than an invented one.

Something similar happens with quality. We list every quality in the file, with its number of rows, to see whether they describe a single scale.

In [16]:
# Every quality published, with its number of rows.
print(f"{raw['Calidad'].nunique()} distinct qualities were found.")
show(raw["Calidad"].value_counts().rename_axis("quality").to_frame("rows"))

55 distinct qualities were found.


,rows
quality,
Primera,"76,248"
Segunda,"27,045"
Especial,"12,543"
Tercera,"6,611"
1a (cosecha),"4,086"
1a (guarda),"4,023"
Extra,"2,901"
1a amarillo,"2,759"
1a nueva(o),"2,092"


**Anomaly 8 · Qualities do not form a scale.** There are 55, and they mix different criteria: grade (`Primera`), size (`Calibre 80`), ripeness (`Pintón`) and harvest (`1a (guarda)`). Even the spelling changes: `1a nueva(o)` has no space and `3a nueva (o)` has one. The file does not say whether `Especial` ranks above `Extra`, or how a size compares with a grade. We keep the qualities as ODEPA writes them and always compare within one quality, without ranking them from best to worst.

In the variety column, the most frequent value is not a variety. We measure how many rows carry it and in which products.

In [17]:
# The most frequent variety label and the products that carry it most.
top = raw["Variedad / Tipo"].value_counts().index[0]
unknown = raw[raw["Variedad / Tipo"] == top]
print(
    f"{number(len(unknown))} rows with the variety «{top}» were found ({round(len(unknown) / len(raw) * 100, 1)} % of the file)."
)
show(unknown["Producto"].value_counts().head(5).rename_axis("product").to_frame("rows"))

56,606 rows with the variety «Sin especificar» were found (36.3 % of the file).


,rows
product,
Limón,"7,886"
Cebolla,"5,869"
Frutilla,"3,578"
Zanahoria,"2,917"
Brócoli,"2,860"


**Anomaly 9 · Variety «Sin especificar».** 36.3 % of the rows, 56,606, state no variety, and lemon is the product with the most rows like that. The label may lump varieties with different prices together. We treat it as one more variety; we neither split it nor fill it in, because the file does not tell which variety it was.

That leaves the origin. We classify each value by its level, region, province or other, and look at the ones that do not fit.

In [18]:
# The level of each origin: region, province or other.
level = raw["Origen"].str.split().str[0].where(raw["Origen"].str.startswith(("Provincia", "Región")), "Other")
print(
    f"{number(int((level == 'Región').sum()))} origins at region level, "
    f"{number(int((level == 'Provincia').sum()))} at province level and {number(int((level == 'Other').sum()))} "
    "at another level were found."
)
show(raw.loc[level == "Other", "Origen"].value_counts().head(7).rename_axis("origin").to_frame("rows"))

83,346 origins at region level, 53,411 at province level and 19,100 at another level were found.


,rows
origin,
Perú,"6,798"
Ecuador,"6,763"
China,"2,266"
Brasil,"1,461"
Cabildo,673
Limache,441
Paine,144


**Anomaly 10 · Origin mixes levels.** There are regions, provinces, municipalities like `Cabildo` or `Limache`, and countries like `Perú`, `Ecuador` or `Brasil`. Some levels overlap, such as `Región Metropolitana` and `Provincia de Santiago`, so prices cannot be added up by origin without choosing one level. Since the analysis is by region of sale and not of production, origin takes part in no calculation. We keep it in the raw layer, in case it is ever needed.

**Step 6 · Does one product have prices of different scales?**

This is the longest step, because this is where the reading of each unit gets decided. The idea is simple: if a reading is right, the price per kilo of a product has to look alike in all its packages. We start by seeing which units have the highest prices.

In [19]:
# Median price of each unit: the most expensive first.
scale = raw.groupby(UNIT)["avg"].agg(rows="size", median_price="median").sort_values("median_price", ascending=False)
print(
    f"The median price of a typical form is ${number(scale['median_price'].median())}; these are the six most expensive."
)
show(scale.head(6).rename_axis("unit"), mark=["median_price"])

The median price of a typical form is $12,000; these are the six most expensive.


,rows,median_price
unit,,
$/bins (420 kilos),684,"305,278"
$/bins (450 kilos),"2,271","275,000"
$/bins (400 kilos),"5,934","250,000"
$/bins (350 kilos),2,"195,000"
$/caja 30 kilos,2,"65,000"
$/malla 50 unidades,22,"42,500"


Bins are far above everything else. To understand why, we take the product with the most rows in bins and take the price of each of its packages to kilo, using the weight the text states.

In [20]:
# The weight the text of each unit states (for example, 400 in «$/bins (400 kilos)»).
def weight(unit):
    found = re.search(r"(\d+(?:,\d+)?) kilos", unit)
    return float(found.group(1).replace(",", ".")) if found else float("nan")


raw["weight"] = raw[UNIT].map({unit: weight(unit) for unit in raw[UNIT].unique()})
packaged = raw[UNIT].str.contains(r"^\$/(?:(?:bandeja|caja|malla|saco) \d|bins \(\d)")
bins = raw[raw[UNIT].str.contains("bins")]
product = bins["Producto"].value_counts().index[0]
detail = (
    raw[(raw["Producto"] == product) & packaged]
    .groupby(UNIT)
    .agg(rows=("avg", "size"), median_price=("avg", "median"), kilos=("weight", "first"))
)
detail["price_per_kilo"] = detail["median_price"] / detail["kilos"]
print(f"The product with the most rows in bins is {product}; these are its packages with at least 30 rows.")
display(
    show(
        detail[detail["rows"] >= 30].sort_values("median_price").rename_axis("unit"),
        mark=["price_per_kilo"],
    )
)
show(bins[bins["Producto"] == product].head(3)[["Fecha", UNIT, "Volumen", "Precio promedio"]])

The product with the most rows in bins is Manzana; these are its packages with at least 30 rows.


,rows,median_price,kilos,price_per_kilo
unit,,,,
$/bandeja 15 kilos granel,"3,640","12,000",15,800
$/caja 16 kilos empedrada,432,"12,000",16,750
$/caja 18 kilos granel,271,"14,000",18,778
$/bandeja 18 kilos granel,698,"15,000",18,833
$/caja 15 kilos empedrada,596,"17,500",15,"1,167"
$/caja 18 kilos embalada,"1,162","23,000",18,"1,278"
$/caja 20 kilos granel,42,"25,300",20,"1,265"
$/bins (400 kilos),"3,180","260,000",400,650


Fecha,Unidad de comercializacion,Volumen,Precio promedio
2026-01-14,$/bins (400 kilos),10,"400000,0000"
2026-01-15,$/bins (400 kilos),20,"375000,0000"
2026-01-16,$/bins (400 kilos),10,"350000,0000"


**Anomaly 11 · The price of a bin is for one bin.** Bins cost between $250,000 and $305,278, while the trays and boxes of apples, the product with the most rows in bins, cost between $12,000 and $25,300. Per kilo, however, apples cost about the same in every package: $650 in the 400-kilo bin and between $750 and $1,278 in trays and boxes. The price gap is explained by a bin holding 400 kilos. One doubt remains: a row says volume 10 and price $400,000; is that the price of one bin or of all ten? If it were for ten, a kilo would cost $100, between seven and thirteen times less than in a tray, which fits nothing. The price is for a single bin, and we divide it by the 400 kilos it holds.

Some units say «$/kilo» and name a box at the same time, like «$/kilo (en caja de 17 kilos)». Is the price for one kilo or for the box? We test both readings against the price per kilo of the same product in trays, boxes, mesh bags and sacks. The right one is the one that stays close to 1.

In [21]:
# Reference price per kilo of each product: its trays, boxes, mesh bags, sacks and bins with a weight in the text.
reference = (raw.loc[packaged, "avg"] / raw.loc[packaged, "weight"]).groupby(raw.loc[packaged, "Producto"]).median()
inside = raw[raw[UNIT].str.startswith("$/kilo (en ")]
readings = (
    inside.groupby(["Producto", UNIT])
    .agg(rows=("avg", "size"), published=("avg", "median"), kilos=("weight", "first"))
    .reset_index()
    .join(reference.rename("reference"), on="Producto")
)
readings["if_per_kilo"] = readings["published"] / readings["reference"]
readings["if_per_box"] = readings["published"] / readings["kilos"] / readings["reference"]
print("Published price against the reference per kilo of each product, under the two possible readings.")
show(readings[readings["rows"] >= 30], mark=["if_per_kilo", "if_per_box"])

Published price against the reference per kilo of each product, under the two possible readings.


Producto,Unidad de comercializacion,rows,published,kilos,reference,if_per_kilo,if_per_box
Chirimoya,$/kilo (en caja de 15 kilos),139,"3,160",15,"2,250",1.4,0.09
Naranja,$/kilo (en caja de 20 kilos),76,950,20,533,1.78,0.09
Palta,$/kilo (en bandeja de 18 kilos),706,"3,500",18,"3,065",1.14,0.06
Palta,$/kilo (en caja de 15 kilos),588,"3,000",15,"3,065",0.98,0.07
Palta,$/kilo (en caja de 17 kilos),"6,539","2,800",17,"3,065",0.91,0.05


**Anomaly 12 · «$/kilo (en caja de 17 kilos)» is a price per kilo.** Read as a price per kilo, it lands between 0.91 and 1.78 times the reference. Read as the price of the box, it drops to between 0.05 and 0.09. For avocado in the 17-kilo box it is 0.91 against 0.05. Only the per-kilo reading makes sense; the box states how much each package counted in the volume weighs. This is our inference: the data back it, although ODEPA does not state it.

The «Volumen» column does not say what it measures either. If it were kilos, a row of 400-kilo bins should show at least 400. For each form with a weight, we measure what share of its rows has a volume below the weight of its package.

In [22]:
# Percentage of rows whose volume is below the weight of the package.
weighted = raw[packaged]
below = (
    weighted.assign(below=weighted["volume"] < weighted["weight"])
    .groupby(UNIT)
    .agg(rows=("below", "size"), pct=("below", "mean"))
)
below["pct"] = (below["pct"] * 100).round()
bins_rows = weighted[weighted[UNIT].str.contains("bins")]
print(
    f"{number(len(bins_rows))} rows in bins were found; in "
    f"{number((bins_rows.volume < bins_rows.weight).sum())} of them the volume is below the weight of the package."
)
print("Percentage of rows with a volume below the weight of the package, by form.")
show(
    below[below["rows"] >= 100]
    .sort_values("pct", ascending=False)
    .head(6)
    .rename(columns={"pct": "% with lower volume"})
    .rename_axis("unit"),
    mark=["% with lower volume"],
)

8,891 rows in bins were found; in 8,891 of them the volume is below the weight of the package.
Percentage of rows with a volume below the weight of the package, by form.


,rows,% with lower volume
unit,,
$/bins (450 kilos),"2,271",100
$/bins (420 kilos),684,100
$/bins (400 kilos),"5,934",100
$/caja 25 kilos,498,24
$/caja 18 kilos,"4,076",1
$/caja 24 kilos,508,1


**Anomaly 13 · Volume is counted in packages.** In bins, all 8,891 rows have a volume below the weight of the package. No other form comes close: the highest, the 25-kilo box, reaches 24 %. If volume were kilos, none of those rows would fill a single bin. We read it as a number of packages, except in the forms that say «(volumen en unidades)», and the price of the week is weighted by what each package holds. This is an inference too: it matches the data and ODEPA's bulletin, which states volume per package, but the file does not document it.

Four forms warn in their own text that volume goes «en unidades». We list them with their products and check whether the volume is a multiple of the pack size.

In [23]:
# Pack size the text states: 100 for «cien», the number before «unidades», or 1.
def pack_size(unit):
    if "cien" in unit:
        return 100
    found = re.search(r"(\d+) unidades", unit)
    return int(found.group(1)) if found else 1


flagged = raw[raw[UNIT].str.contains("volumen")]
flagged = flagged.assign(multiple=flagged["volume"] % flagged[UNIT].map(pack_size) == 0)
by_unit = flagged.groupby(UNIT).agg(
    rows=("avg", "size"),
    products=("Producto", lambda s: ", ".join(s.value_counts().index[:2])),
    pct_multiple=("multiple", "mean"),
)
by_unit["pct_multiple"] = (by_unit["pct_multiple"] * 100).round()
print(f"{len(by_unit)} forms that state «(volumen en unidades)» were found.")
show(by_unit.rename(columns={"pct_multiple": "% multiple of the pack"}).rename_axis("unit"))

4 forms that state «(volumen en unidades)» were found.


,rows,products,% multiple of the pack
unit,,,
$/cien unidades (volumen en unidades),148,Rabanito,100
$/kilo (volumen en unidades),"2,599","Zapallo, Sandia",100
$/paquete 10 unidades (volumen en unidades),168,Cebolla,100
$/paquete 20 unidades (volumen en unidades),730,"Cebolla, Ajo",99


**Anomaly 14 · Price per kilo, volume in pieces.** The main form is «$/kilo (volumen en unidades)», with 2,599 rows of pumpkin and watermelon. The price is per kilo, but the volume counts whole pieces, and a pumpkin does not always weigh the same. In the other three forms the volume is indeed a multiple of the pack size. Today each piece is weighted as if it weighed one kilo. We decided to keep it and write it into the rule, because it does not change the price of each day, only how much that day weighs within the week, and making up a weight per piece would be worse.

«Cien» is left. We compute the price per unit under both possible readings and compare it with what things sold per unit cost in the same subsector.

In [24]:
# The price per unit of «cien» under the two readings, against what is sold per unit in its subsector.
hundred = raw[raw[UNIT].str.contains("cien")]
subsector = hundred["Subsector"].iloc[0]
per_unit = raw[(raw[UNIT] == "$/unidad") & (raw["Subsector"] == subsector)]["avg"]
print("Price per unit of «cien» under each reading, together with references from the subsector.")
show(
    pd.DataFrame(
        {
            "price per unit": [
                hundred["avg"].median(),
                hundred["avg"].median() / 100,
                per_unit.median(),
                per_unit.quantile(0.9),
            ]
        },
        index=[
            f"{hundred['Producto'].iloc[0]}, if «cien» is one unit",
            f"{hundred['Producto'].iloc[0]}, if «cien» is 100 units",
            "reference: median of what is sold per unit",
            "reference: 90th percentile",
        ],
    ).rename_axis("reading")
)

Price per unit of «cien» under each reading, together with references from the subsector.


,price per unit
reading,
"Rabanito, if «cien» is one unit","3,500"
"Rabanito, if «cien» is 100 units",35
reference: median of what is sold per unit,"1,000"
reference: 90th percentile,"2,000"


**Anomaly 15 · «Cien» means a hundred units.** The form has 148 rows, all of radish, with a median of $3,500. If that were the price of one radish, it would cost more than 90 % of what is sold per unit in its subsector, whose 90th percentile is $2,000. Read as a hundred units, a radish costs $35, which is reasonable. We divide the price by 100. Like 12 and 13, this is an inference the data back, although ODEPA does not state it.

**Step 7 · Are there prices out of the ordinary?**

A price on another scale is usually a typing error or a wrongly recorded package, and a single one is enough to move the average of the week. We compare each price with the median of its group (same product, variety, quality, unit and market) and list those more than five times away.

In [25]:
# How many times each price is the median of its group; only groups of at least 10 rows.
group = raw.groupby(["Producto", "Variedad / Tipo", "Calidad", UNIT, "Mercado"])["avg"]
raw["group_median"] = group.transform("median")
raw["times"] = raw["avg"] / raw["group_median"]
enough = raw[group.transform("size") >= 10]
far = enough[(enough["times"] > 5) | (enough["times"] < 0.2)]
print(
    f"{len(far)} rows more than five times away from the median of their group were found; "
    "these are the three lowest and the three highest."
)
show(
    pd.concat([far.nsmallest(3, "times"), far.nlargest(3, "times")])[
        ["Fecha", "Mercado", "Producto", UNIT, "Precio promedio", "group_median", "times"]
    ].rename(columns={"group_median": "group median", "times": "times the median"}),
    mark=["times the median"],
)

96 rows more than five times away from the median of their group were found; these are the three lowest and the three highest.


Fecha,Mercado,Producto,Unidad de comercializacion,Precio promedio,group median,times the median
2026-05-08,Vega Central Mapocho de Santiago,Apio,$/docena de matas,"600,0000","9,000",0.07
2026-03-11,Mercado Mayorista Lo Valledor de Santiago,Cebolla,$/caja 20 kilos,"500,0000","6,000",0.08
2026-01-07,Mercado Mayorista Lo Valledor de Santiago,Poroto verde,$/saco 25 kilos,"2000,0000","22,000",0.09
2026-03-19,Vega Modelo de Temuco,Pera,$/bandeja 18 kilos granel,"320000,0000","14,250",22.46
2026-03-11,Vega Monumental Concepción,Coliflor,$/unidad,"10000,0000",800,12.5
2026-07-20,Vega Modelo de Temuco,Sandia,$/kilo (volumen en unidades),"15000,0000","1,400",10.71


**Anomaly 16 · Prices out of scale.** 96 rows show up. The most extreme is a pear in an 18-kilo tray published at $320,000, when that tray usually costs $14,250, the median of its group; the figure looks more like the price of a bin. At the other end, a 20-kilo box of onions appears at $500. The file does not tell whether these are typing errors, packages recorded wrongly or real prices, so we correct no number. The rows are flagged as outliers (the app compares them with the rest of their own week) and left out of the indicators, without being deleted. This pear, for instance, is flagged in the database.

**Step 8 · Do packages change from one week to the next?**

The last step looks at time. If a product is sold one week in an 18-kilo tray and the next in a 10-kilo tray, its price may seem to rise without anything having changed. For each product, variety, quality and region we compare the packages of consecutive weeks and look for the cases where they share none.

In [26]:
# Packages and price per kilo of each week, and those of the previous week of the same series.
raw["per_kilo"] = (raw["avg"] / raw["weight"]).where(packaged)
raw["week"] = pd.to_datetime(raw["Fecha"]).dt.to_period("W-SUN").dt.start_time
series = ["Producto", "Variedad / Tipo", "Calidad", "Region"]
weekly = (
    raw.groupby(series + ["week"])
    .agg(units=(UNIT, lambda s: " · ".join(sorted(set(s)))), price=("per_kilo", "mean"))
    .reset_index()
    .sort_values(series + ["week"])
)
previous = weekly.groupby(series)
weekly["units_before"], weekly["price_before"] = previous["units"].shift(), previous["price"].shift()
weekly["week_before"] = previous["week"].shift()
consecutive = (weekly["week"] - weekly["week_before"]).dt.days == 7
shared = pd.Series(
    [
        len(set(a.split(" · ")) & set(b.split(" · "))) if isinstance(b, str) else -1
        for a, b in zip(weekly["units"], weekly["units_before"])
    ],
    index=weekly.index,
)
cases = weekly[consecutive & (shared == 0)].copy()
cases["jump"] = (cases["price"] / cases["price_before"] - 1) * 100
latest = cases[cases["week"] == cases["week"].max()].sort_values("jump", ascending=False)
print(
    f"{number(consecutive.sum())} pairs of consecutive weeks were compared; {number(len(cases))} share no "
    f"package and {len(latest)} of them belong to the latest week."
)
show(
    latest.head(5)[series + ["units_before", "units", "price_before", "price", "jump"]].rename(
        columns={
            "units_before": "package before",
            "units": "package now",
            "price_before": "price per kilo before",
            "price": "price per kilo now",
            "jump": "apparent jump (%)",
        }
    ),
    mark=["apparent jump (%)"],
)

32,524 pairs of consecutive weeks were compared; 827 share no package and 20 of them belong to the latest week.


Producto,Variedad / Tipo,Calidad,Region,package before,package now,price per kilo before,price per kilo now,apparent jump (%)
Kiwi,Hayward,Especial,Región del Maule,$/bandeja 18 kilos,$/bandeja 10 kilos,"1,111","1,900",71
Naranja,Navel Late,Especial,Región de La Araucanía,$/bandeja 15 kilos granel,$/caja 15 kilos granel,"1,075","1,333",24
Manzana,Pink Lady,Primera,Región del Biobío,$/bandeja 15 kilos granel,$/caja 16 kilos empedrada,867,938,8.17
Cebolla,Sin especificar,1a (guarda),Región de La Araucanía,$/malla 15 kilos,$/malla 18 kilos,"1,031","1,111",7.76
Manzana,Granny Smith,Especial,Región del Biobío,$/bandeja 15 kilos granel,$/caja 16 kilos empedrada,"1,067","1,109",4


**Anomaly 17 · Package changes that look like price changes.** Of 32,524 pairs of consecutive weeks, 827 share no package. In the latest week there are 20, and the one that jumps the most is Hayward Especial kiwi in Maule: it goes from an 18-kilo tray, at $1,111 a kilo, to a 10-kilo tray, at $1,900. It looks like a 71 % rise, but what changed was the package. That is why every variation is computed only between prices of the same package in both weeks, and if there is none in common, the price is left out of the comparison. With this rule, in the app the alert for this kiwi went down from +25.6 % to +8.8 % and stopped being an alert.

**In short**

The procedure found 17 anomalies. This table gathers what the app does with each one.

| # | Anomaly | Treatment in the app | State |
|---|---|---|---|
| 1 | Numbers published as text, with a decimal comma | Converts to numbers and keeps the original text | Treated |
| 2 | Days and markets without data | Fills nothing in; a missing week stops the load | Treated |
| 3 | Average price different from the midpoint | Uses the average published by ODEPA | Treated |
| 4 | Minimum and maximum far apart | Uses only the average price | Treated |
| 5 | Repeated rows that differ in origin | Keeps every row | Treated |
| 6 | Unit of sale written in 148 forms | List reviewed by hand; a new form stops the load | Treated |
| 7 | Units without a defined weight | Publishes without a price per kilo, flagged, not deleted | Treated |
| 8 | Qualities without a defined order | Compares within the same quality | Treated |
| 9 | Variety «Sin especificar» | Treats it as one more variety | Treated |
| 10 | Origin at different levels | Does not use it in calculations | Treated |
| 11 | Price per package, not for the total volume | Divides the price by what the package holds | Treated |
| 12 | «$/kilo (en caja de 17 kilos)» | Treats it as a price per kilo | Treated · inference |
| 13 | Volume expressed in packages | Weights the week by what each package holds | Treated · inference |
| 14 | «$/kilo (volumen en unidades)» | Weights each piece as 1 kilo | Treated |
| 15 | «$/cien unidades» | Treats it as 100 units | Treated · inference |
| 16 | Prices out of scale | Flags them as outliers, not deleted | Treated |
| 17 | Change of package between weeks | Compares only the same package | Treated |

All 17 are treated. Three of them, 12, 13 and 15, rest on inferences: the data back them, although ODEPA does not state them.